# 01 — Data inspection

Open a single trial, list the variables, plot `ce_foot` / `ce_torso`, and
check that axes look right.

The dataset's `.mat` files are MATLAB v5 — readable with `scipy.io.loadmat`
(not h5py). Arrays come out as (doppler, time) directly; no transpose
needed. The project's `src.data_loader.load_trial` wraps this.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.io as sio

plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.figsize"] = (10, 4)

from src.data_loader import DATA_ROOT, load_trial, list_keys, quick_shape_info
from src import radar_params as P


## 1. List variables in one file (no data load)

In [ ]:
TRIAL = DATA_ROOT / "fisc_005" / "test1" / "trial1" / "stft_data.mat"
print(f"File size: {TRIAL.stat().st_size / 1e6:.1f} MB\n")

for k in sorted(list_keys(TRIAL)):
    shape = quick_shape_info(TRIAL, k)
    print(f"  {k:22s}  shape={shape}")


## 2. Load via `src.data_loader.load_trial`

Result is already (doppler, time).

In [ ]:
trial = load_trial(TRIAL, representation="ce")
for k, v in trial.items():
    if hasattr(v, "shape"):
        print(f"  {k:10s}  shape={v.shape}  dtype={v.dtype}")
    else:
        print(f"  {k:10s}  {v}")


## 3. Visualise `ce_foot` and `ce_torso`

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for ax, key in zip(axes, ["ce_foot", "ce_torso"]):
    spec = np.log1p(trial[key])
    im = ax.imshow(spec, aspect="auto", origin="lower",
                   extent=[trial["t_axis"].min(), trial["t_axis"].max(),
                           trial["doppler"].min(), trial["doppler"].max()],
                   cmap="magma")
    ax.set_ylabel("Doppler [Hz]")
    ax.set_title(f"{key}   shape={trial[key].shape}")
    fig.colorbar(im, ax=ax, fraction=0.03, pad=0.01, label="log1p(value)")
axes[-1].set_xlabel("Time [s]")
fig.suptitle(str(TRIAL.relative_to(DATA_ROOT)), fontsize=10)
plt.tight_layout()
plt.show()


## 4. Check axes against the paper

The paper's STFT has 20 Hz **intrinsic resolution** (1 / 50 ms window). The
dataset here stores 320 bins over ±800 Hz → **5 Hz bin spacing**, i.e. a
4× zero-padded FFT. This is denser than what the paper analyses but does
not add new information; useful for finer visualisation, and the QC code
checks against 5 Hz, not 20 Hz.


In [ ]:
t, d = trial["t_axis"], trial["doppler"]
print(f"Time axis    n={t.size:6d}  range=[{t.min():.2f}, {t.max():.2f}] s  dt≈{np.median(np.diff(t))*1000:.2f} ms")
print(f"Doppler axis n={d.size:6d}  range=[{d.min():.1f}, {d.max():.1f}] Hz  Δf={np.median(np.diff(d)):.2f} Hz")
print()
print(f"  paper intrinsic resolution: {P.DOPPLER_RESOLUTION_HZ:.0f} Hz")
print(f"  dataset bin spacing:        {P.DOPPLER_BIN_SPACING_HZ:.0f} Hz  (4× zero-padded)")
print()
print(f"λ = {P.WAVELENGTH_M*1000:.2f} mm")
print(f"Max |Doppler| in this trial: {abs(d).max():.0f} Hz → v_max ≈ {P.doppler_to_velocity_m_s(abs(d).max()):.2f} m/s")


## 5. Compare `ce_foot` to `|stft_foot|`

The paper does *not* describe a contrast-enhancement step, so `ce_*` is a downstream addition. Look for differences in dynamic range and contrast.

In [ ]:
trial_full = load_trial(TRIAL, representation="both")
stft_abs = np.abs(trial_full["stft_foot"])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, spec, title in [
    (axes[0], np.log1p(stft_abs),       "log1p(|stft_foot|)"),
    (axes[1], np.log1p(trial["ce_foot"]),"log1p(ce_foot)"),
]:
    ax.imshow(spec, aspect="auto", origin="lower", cmap="magma")
    ax.set_title(title)
plt.tight_layout(); plt.show()

print(f"|stft_foot|: min={stft_abs.min():.3g}  max={stft_abs.max():.3g}  mean={stft_abs.mean():.3g}")
print(f"ce_foot:     min={trial['ce_foot'].min():.3g}  max={trial['ce_foot'].max():.3g}  mean={trial['ce_foot'].mean():.3g}")


### What to check before moving on

1. Section 3 panels show **time on X, Doppler on Y**. If not, check `src/data_loader.py`.
2. Section 4 confirms `Δf ≈ 5 Hz` and 320 bins (or close to it).
3. `ce_foot` and `|stft_foot|` look qualitatively similar — `ce_*` should mainly *enhance contrast*, not invent structure.
